In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
# import xgboost as xgb
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig
from ax.generation_strategy.center_generation_node import CenterGenerationNode
from ax.generation_strategy.transition_criterion import MinTrials
from ax.generation_strategy.generation_strategy import GenerationStrategy
from ax.generation_strategy.generation_node import GenerationNode
from ax.generation_strategy.model_spec import GeneratorSpec
from ax.modelbridge.registry import Generators
from gpytorch.kernels import MaternKernel
from botorch.models import SingleTaskGP
from botorch.models.transforms.input import Warp
from botorch.models.map_saas import AdditiveMapSaasSingleTaskGP
from ax.utils.stats.model_fit_stats import MSE
from ax.models.torch.botorch_modular.surrogate import SurrogateSpec, ModelConfig
from botorch.acquisition.logei import qLogNoisyExpectedImprovement
import seaborn

In [2]:
o = 27
n = 1
s = o-n
sampler = Sampler_class()
Parameters_lis = [
    RangeParameterConfig(name="s1", parameter_type="float", bounds=tuple([0,1])),
    RangeParameterConfig(name="s2", parameter_type="float", bounds=tuple([0,1])),
    RangeParameterConfig(name="b1", parameter_type="float", bounds=tuple([0,1])),
    ]

In [3]:
client = Client()
gp_model = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/Modelling/ModelMk4.json")
gp_model.get_next_trials(max_trials=1)
def SurrogateModelOfReality(s1, s2, b1):
    y_pred = gp_model.predict([{"s1":s1,"s2":s2,"b1":b1}])[0]["t1"][0]
    return np.float64(y_pred)

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


In [4]:
y_max_lis = []

for i in range(100):
    client = Client()
    parameters = [
        RangeParameterConfig(
            name="s1", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="s2", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="b1", parameter_type="float", bounds=(0, 1)
        ),
    ]
    client.configure_experiment(parameters=parameters)
    def construct_generation_strategy(
        generator_spec: GeneratorSpec, node_name: str,
    ) -> GenerationStrategy:
        """Constructs a Center + Sobol + Modular BoTorch `GenerationStrategy`
        using the provided `generator_spec` for the Modular BoTorch node.
        """
        botorch_node = GenerationNode(
            node_name=node_name,
            model_specs=[generator_spec],
        )
        return GenerationStrategy(
            name=f"{node_name}",
            nodes=[botorch_node]
        )

    # Let's construct the simplest version with all defaults.
    construct_generation_strategy(
        generator_spec=GeneratorSpec(model_enum=Generators.BOTORCH_MODULAR),
        node_name="Modular BoTorch",
    )

    surrogate_spec = SurrogateSpec(
        model_configs=[
            # Select between two models:
            # An additive mixture of relatively strong SAAS priors with input Warping.
            # A relatively vanilla GP with a Matern kernel.
            ModelConfig(
                botorch_model_class=SingleTaskGP,
                covar_module_class=MaternKernel,
                covar_module_options={"nu": 2.5},
            ),
        ],
        eval_criterion=MSE,  # Select the model to use as the one that minimizes mean squared error.
        allow_batched_models=False,  # Forces each metric to be modeled with an independent BoTorch model.
        # If we wanted to specify different options for different metrics.
        # metric_to_model_configs: dict[str, list[ModelConfig]]
    )

    generator_spec = GeneratorSpec(
        model_enum=Generators.BOTORCH_MODULAR,
        model_kwargs={
            "surrogate_spec": surrogate_spec,
            "botorch_acqf_class": qLogNoisyExpectedImprovement,
            # Can be used for additional inputs that are not constructed
            # by default in Ax. We will demonstrate below.
            "acquisition_options": {},
        },
        # We can specify various options for the optimizer here.
        model_gen_kwargs = {
            "model_gen_options": {
                "optimizer_kwargs": {
                    "num_restarts": 20,
                    "sequential": False,
                    "options": {
                        "batch_limit": 5,
                        "maxiter": 200,
                    },
                },
            },
        }
    )

    generation_strategy = construct_generation_strategy(
        generator_spec=generator_spec,
        node_name="BoTorch w/ Model Selection",
    )
    generation_strategy

    client.set_generation_strategy(
        generation_strategy=generation_strategy,
    )

    metric_name = "t1" # this name is used during the optimization loop in Step 5
    objective = f"{metric_name}" # minimization is specified by the negative sign

    client.configure_optimization(objective=objective)

    X = sampler.three.PseudorandomSampler3D_func(n,Parameters_lis).T

    for array in X:
        my_parameters = {"s1": array[0], "s2": array[1], "b1": array[2]}
        trial_index = client.attach_trial(parameters=my_parameters)
        client.complete_trial(trial_index=trial_index,raw_data={"t1": SurrogateModelOfReality(**my_parameters)})

    for _ in range(s): # Run 10 rounds of trials
        # We will request three trials at a time in this example
        trials = client.get_next_trials(max_trials=1)

        for trial_index, parameters in trials.items():
            s1 = parameters["s1"]
            s2 = parameters["s2"]
            b1 = parameters["b1"]

            result = SurrogateModelOfReality(s1, s2, b1)

            # Set raw_data as a dictionary with metric names as keys and results as values
            raw_data = {metric_name: result}

            # Complete the trial with the result
            client.complete_trial(trial_index=trial_index, raw_data=raw_data)
    # print(client.summarize())
    print(f"Trial {i} =========================================")
    y_max = np.max(np.array(client.summarize().t1))
    print(y_max)
    y_max_lis.append(y_max)
    print()

y_max_arr = np.array(y_max_lis)
print(y_max_arr)

[WARNING 06-25 13:53:17] ax.modelbridge.transforms.standardize_y: Outcome t1 is constant, within tolerance.
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/ax/modelbridge/cross_validation.py:382: UserWarning: Encountered exception in computing model fit quality: Outcome `t1` was not observed.
  warn("Encountered exception in computing model fit quality: " + str(e))
[WARNING 06-25 13:56:21] ax.modelbridge.transforms.standardize_y: Outcome t1 is constant, within tolerance.


Trial 0 =========================================
12.895139279623537



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/ax/modelbridge/cross_validation.py:382: UserWarning: Encountered exception in computing model fit quality: Outcome `t1` was not observed.
  warn("Encountered exception in computing model fit quality: " + str(e))
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
[WARNING 06-25 13:57:21] ax.modelbridge.transforms.standardize_y: Outcome t1 is constant, within tolerance.


Trial 1 =========================================
16.11073650426021



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/ax/modelbridge/cross_validation.py:382: UserWarning: Encountered exception in computing model fit quality: Outcome `t1` was not observed.
  warn("Encountered exception in computing model fit quality: " + str(e))
[WARNING 06-25 13:58:17] ax.modelbridge.transforms.standardize_y: Outcome t1 is constant, within tolerance.


...

Trial 97 =========================================
14.476853118598704



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/ax/modelbridge/cross_validation.py:382: UserWarning: Encountered exception in computing model fit quality: Outcome `t1` was not observed.
  warn("Encountered exception in computing model fit quality: " + str(e))
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
[WARNING 06-25 15:29:42] ax.modelbridge.transforms.standardize_y: Outcome t1 is constant, within tolerance.


Trial 98 =========================================
16.11864796003604



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/ax/modelbridge/cross_validation.py:382: UserWarning: Encountered exception in computing model fit quality: Outcome `t1` was not observed.
  warn("Encountered exception in computing model fit quality: " + str(e))


Trial 99 =========================================
13.069412817774383

[12.89513928 16.1107365  15.56807527 12.49349217 16.19708905 13.1324075
 15.38802759 15.35684534 16.11609592 16.26327339 15.39970374 15.91984405
 15.70607538 15.92922201 17.29459339 15.66277532 16.21922434 14.19740394
 13.42407382 15.92922201 14.5091454  16.31422499 14.30946012 15.97089031
 17.59566294 18.80395879 16.04432846 13.0293684  15.92922201 15.37338071
 15.92922201 14.61650888 13.43223914 16.04895474 15.13637159 15.92922201
 14.46069599 15.72083224 14.60727915 18.79112832 17.34125112 14.63486748
 12.99609326 15.34128872 14.77314991 16.24252826 15.78929124 15.92922201
 14.2820476  17.52748898 16.21980288 16.4489126  15.22672553 16.97508843
 14.36186945 15.39048235 14.85452597 14.58881287 16.23295204 16.22600588
 16.22446595 15.39829966 14.27224676 17.17120086 17.58271534 14.15295953
 15.92922201 15.38957839 16.22581841 15.2734469  15.29250099 15.39558786
 18.74083164 13.63353448 16.11367438 16.24280755 14.12

In [5]:
print(f"Max = {np.max(y_max_arr)}")
print(f"Avg = {np.average(y_max_arr)}")
print(f"Std = {np.std(y_max_arr)}")

Max = 18.803958789664307
Avg = 15.44395773172856
Std = 1.3706535669011402


In [6]:
print(y_max_arr.tolist())

[12.895139279623537, 16.11073650426021, 15.568075273839936, 12.49349216985139, 16.197089049294487, 13.132407499932695, 15.388027586370747, 15.356845337825366, 16.116095915008728, 16.263273389427102, 15.399703740595417, 15.919844054379313, 15.706075381303169, 15.929222010399386, 17.29459339192895, 15.662775321779012, 16.219224341728285, 14.19740393839031, 13.424073822830154, 15.929222010399386, 14.509145399488315, 16.31422498613562, 14.309460115178197, 15.970890309483583, 17.595662935094897, 18.803958789664307, 16.044328461414217, 13.029368397611885, 15.929222010399386, 15.373380710056473, 15.929222010399386, 14.61650888158844, 13.432239141518652, 16.04895474065444, 15.13637158558174, 15.929222010399386, 14.460695994234944, 15.720832236042114, 14.607279145113175, 18.791128324430268, 17.341251117476713, 14.634867476218865, 12.99609325966306, 15.34128871551261, 14.773149906730676, 16.24252826151293, 15.789291240771247, 15.929222010399386, 14.282047595292804, 17.527488979933956, 16.2198028

In [7]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/SequentialTestingOfSamplingTechnique/DataGenerated/BOpt-EI-9,27,1-1.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
latestdf = pd.DataFrame(y_max_arr)
newdf = pd.concat(objs=[loadeddf,latestdf],axis=0)
newdf = newdf.reset_index(drop=True)
pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)